# Task 6: High-Throughput Asynchronous RAG Pipeline with Cross-Encoder Reranking

## Objective

To build an asynchronous Retrieval-Augmented Generation (RAG) pipeline using bi-encoder retrieval, FAISS vector search, cross-encoder reranking, and concurrent query processing.

## Technologies / Tools Used

- Python
- FastAPI
- LangChain
- FAISS
- PyTorch
- SentenceTransformers
- Asyncio
- Google Colab

## RAG Pipeline

\[
Query \rightarrow Bi\text{-}Encoder \rightarrow FAISS \rightarrow Cross\text{-}Encoder \rightarrow LLM
\]

The bi-encoder retrieves candidate documents, while the cross-encoder reranks them according to query-document relevance.

In [1]:
# Install required libraries

!pip -q install faiss-cpu sentence-transformers fastapi langchain langchain-community

import asyncio
import time
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from fastapi import FastAPI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Step 1: Create Knowledge Documents

Create a small document collection that will be used as the knowledge base for retrieval.

In [2]:
documents = [
    "Artificial intelligence enables machines to perform tasks that normally require human intelligence.",
    "Machine learning allows systems to learn patterns from data and make predictions.",
    "Deep learning uses neural networks with multiple layers to learn complex representations.",
    "Retrieval augmented generation combines information retrieval with language generation.",
    "FAISS is a library designed for efficient similarity search over dense vectors.",
    "Cybersecurity protects computer systems, networks, applications, and data from attacks."
]

print("Number of documents:", len(documents))

Number of documents: 6


## Step 2: Create Bi-Encoder Embeddings and FAISS Index

Convert documents into vector embeddings and store them in an in-memory FAISS index for fast similarity retrieval.

In [3]:
# Load embedding model

embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Create embeddings

embeddings = embedder.encode(
    documents,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

# Create FAISS index

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(embeddings)

print("FAISS index created.")
print("Vectors indexed:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created.
Vectors indexed: 6


## Step 3: Retrieve and Rerank Documents

Retrieve candidate documents using FAISS and rerank them using a cross-encoder to remove less relevant contexts.

In [4]:
# Load cross-encoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def retrieve_and_rerank(query, top_k=3):

    # First-stage retrieval

    query_vector = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_vector,
        top_k
    )

    candidates = [
        documents[i]
        for i in indices[0]
    ]

    # Second-stage reranking

    pairs = [
        [query, doc]
        for doc in candidates
    ]

    rerank_scores = reranker.predict(
        pairs
    )

    ranked = sorted(
        zip(candidates, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked


query = "What is retrieval augmented generation?"

results = retrieve_and_rerank(query)

for doc, score in results:
    print(f"{score:.4f} -> {doc}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

8.7084 -> Retrieval augmented generation combines information retrieval with language generation.
-11.3204 -> Artificial intelligence enables machines to perform tasks that normally require human intelligence.
-11.3363 -> Machine learning allows systems to learn patterns from data and make predictions.


## Step 4: Build Asynchronous RAG Pipeline

Use asyncio to process multiple client queries concurrently and synthesize responses from the retrieved contexts.

In [5]:
async def synthesize(query, context):

    # Simulated asynchronous LLM synthesis
    await asyncio.sleep(0.01)

    return (
        f"Answer: {query}\n\n"
        f"Relevant context: {context}"
    )


async def rag_query(query):

    results = retrieve_and_rerank(
        query,
        top_k=3
    )

    context = " ".join(
        doc for doc, score in results[:2]
    )

    answer = await synthesize(
        query,
        context
    )

    return answer


async def main():

    queries = [
        "What is RAG?",
        "What is FAISS?",
        "What is machine learning?"
    ]

    start = time.perf_counter()

    responses = await asyncio.gather(
        *(rag_query(q) for q in queries)
    )

    elapsed = time.perf_counter() - start

    for response in responses:
        print(response)
        print("-" * 60)

    print(f"Total time: {elapsed:.4f} seconds")


await main()

Answer: What is RAG?

Relevant context: FAISS is a library designed for efficient similarity search over dense vectors. Deep learning uses neural networks with multiple layers to learn complex representations.
------------------------------------------------------------
Answer: What is FAISS?

Relevant context: FAISS is a library designed for efficient similarity search over dense vectors. Cybersecurity protects computer systems, networks, applications, and data from attacks.
------------------------------------------------------------
Answer: What is machine learning?

Relevant context: Machine learning allows systems to learn patterns from data and make predictions. Artificial intelligence enables machines to perform tasks that normally require human intelligence.
------------------------------------------------------------
Total time: 0.2190 seconds


## Step 5: Create FastAPI RAG Service

Expose the asynchronous RAG pipeline through a FastAPI endpoint.

In [6]:
app = FastAPI(
    title="Async RAG Service"
)


@app.get("/query")
async def query_rag(q: str):

    start = time.perf_counter()

    answer = await rag_query(q)

    latency = time.perf_counter() - start

    return {
        "query": q,
        "answer": answer,
        "latency_seconds": round(
            latency,
            4
        )
    }


print("FastAPI RAG service created.")
print("Endpoint: /query?q=your_question")

FastAPI RAG service created.
Endpoint: /query?q=your_question


## Conclusion

The asynchronous RAG pipeline was successfully implemented using FAISS and SentenceTransformers for bi-encoder retrieval, a Cross-Encoder for reranking, asyncio for concurrent processing, and FastAPI for API-based access. The two-stage retrieval process improves the relevance of the context supplied to the generation stage.